# ECB Interview Scraper

This notebook implements a reproducible Python workflow for collecting and structuring interview material published on the official European Central Bank website. The output is a research-ready dataset containing interview metadata, full text, and extracted question-and-answer segments where available.

## Main outputs
The resulting dataset includes:
- interview date
- speaker
- media outlet
- title
- full interview text
- structured question-and-answer pairs

## Methodological features
Compared with manual collection, this workflow improves reproducibility and consistency through:
- Environment and package imports
- automated URL discovery from ECB interview listing pages
- Date extraction from URL
- Media outlet extraction
- Text cleaning
- Q&A structure extraction
- speaker extraction from page content
- Interview extraction pipeline
- Batch extraction (Running the Scraper)
- Data export
- Q&A long-format dataset

## 1. Environment and package imports

In [3]:
import re
import time
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

HEADERS = {
    "User-Agent": "Mozilla/5.0 (research-scraper; academic use)",
    "Accept-Language": "en"
}

DELAY = 1  # seconds between requests — be polite

## 2. Automated URL Discovery

The ECB interview archive is listed at:

`https://www.ecb.europa.eu/press/pubbydate/html/index.en.html?name_of_publication=Interview`

Because the page uses dynamic loading (infinite scroll), a standard HTTP request would not retrieve the full archive. Selenium is therefore used at this stage to open the page in a headless browser, scroll repeatedly until no additional interview entries are loaded, and then extract the full set of interview links.

Each interview URL follows the general pattern:

`/press/inter/date/{YYYY}/html/ecb.in{YYMMDD}~{hash}.en.html`

This Selenium-based step is used only for URL discovery. Once the archive links have been collected, individual interview pages are processed using `requests` and `BeautifulSoup`.

In [6]:
def discover_interview_urls():
    """
    Discover all English-language interview URLs from the ECB website.
    
    Uses Selenium to open the ECB interview listing page in a headless
    Chrome browser and scrolls down repeatedly until all interviews
    have loaded (the page uses infinite scroll, loading 10-20 items
    per batch). Stops when no new links appear after several attempts.
    
    Returns a deduplicated, sorted list of URLs.
    """
    LISTING_URL = (
        "https://www.ecb.europa.eu/press/pubbydate/html/"
        "index.en.html?name_of_publication=Interview"
    )
    
    # Set up headless Chrome (no visible browser window)
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    
    print("Launching headless Chrome...")
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(LISTING_URL)
    
    # Give the page time to fully render before first scroll
    print("Waiting for initial page load...")
    time.sleep(5)
    
    print("Scrolling to load all interviews...\n")
    
    # Track by LINK COUNT rather than page height
    # (more reliable — page height can plateau temporarily
    #  even when more content is still loading)
    previous_link_count = 0
    no_new_links_count = 0
    scroll_round = 0
    
    while no_new_links_count < 10:  # 10 consecutive scrolls with no new links
        scroll_round += 1
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)  # wait for content to load
        
        # Count interview links currently on the page
        links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/press/inter/date/']")
        current_count = len(links)
        
        if current_count > previous_link_count:
            print(f"  Round {scroll_round}: {current_count} links found")
            no_new_links_count = 0  # reset patience counter
        else:
            no_new_links_count += 1
        
        previous_link_count = current_count
    
    print(f"\n  Scrolling complete after {scroll_round} rounds.")
    print(f"  Total raw links on page: {previous_link_count}")
    
    # Extract all interview links from the fully loaded page
    all_links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/press/inter/date/']")
    
    all_urls = set()
    for link in all_links:
        href = link.get_attribute("href")
        if href and href.endswith(".en.html") and "index" not in href:
            all_urls.add(href)
    
    driver.quit()
    print(f"  Browser closed.")
    
    # Sort newest first
    result = sorted(all_urls, reverse=True)
    return result

In [8]:
print("Discovering interview URLs...\n")
interview_urls = discover_interview_urls()
print(f"\nTotal unique English interview URLs found: {len(interview_urls)}")

Discovering interview URLs...

Launching headless Chrome...
Waiting for initial page load...
Scrolling to load all interviews...

  Round 1: 20 links found
  Round 2: 30 links found
  Round 3: 40 links found
  Round 4: 50 links found
  Round 5: 60 links found
  Round 6: 70 links found
  Round 7: 80 links found
  Round 8: 90 links found
  Round 9: 100 links found
  Round 10: 110 links found
  Round 11: 120 links found
  Round 12: 130 links found
  Round 13: 140 links found
  Round 14: 150 links found
  Round 15: 160 links found
  Round 16: 170 links found
  Round 17: 180 links found
  Round 18: 190 links found
  Round 19: 200 links found
  Round 20: 210 links found
  Round 21: 220 links found
  Round 22: 230 links found
  Round 23: 240 links found
  Round 24: 250 links found
  Round 25: 260 links found
  Round 26: 270 links found
  Round 27: 280 links found
  Round 28: 290 links found
  Round 29: 300 links found
  Round 30: 310 links found
  Round 31: 320 links found
  Round 32: 330 lin

In [9]:
if interview_urls:
    total = len(interview_urls)
    newest = interview_urls[0]
    oldest = interview_urls[-1]

    print(f"\nTotal unique English interview URLs found: {total}")
    print(f"Newest: {newest[:80]}...")
    print(f"Oldest: {oldest[:80]}...")
else:
    print("No interview URLs were discovered.")


Total unique English interview URLs found: 590
Newest: https://www.ecb.europa.eu/press/inter/date/2026/html/ecb.in260323~b893c61107.en....
Oldest: https://www.ecb.europa.eu/press/inter/date/2004/html/sp040618.en.html...


## 3. Date extraction from URL

Interview dates are derived from the ECB URL rather than from page HTML. This is deliberate: ECB page templates have changed over time, but the URL slug consistently encodes the publication date across historical formats. Parsing the date directly from the URL therefore improves stability, avoids unnecessary HTML dependence, and makes failures easier to detect.

The ECB has used multiple URL conventions over the sample period (pre-2018 and post-2018 formats, plus rare edge cases). A single regex is therefore used to handle all observed variants and avoid systematic data loss from partial matching.

In [11]:
def parse_date_from_url(url):
    """
    Extract the date from an ECB interview URL.

    The ECB has used three main URL formats over the years and one rare pattern:
      - sp{YYMMDD}.en.html              e.g. sp120323     (pre-2018)
      - sp{YYMMDD}_{N}.en.html          e.g. sp170324_1   (same-day dupes)
      - ecb.in{YYMMDD}~{hash}.en.html   e.g. ecb.in250902 (2018+)
      - in{YYMMDD}~{hash}.en.html  (rare outlier)

    Returns an ISO date string "YYYY-MM-DD", or "" if no match.
    """
    m = re.search(r"/(?:sp|(?:ecb\.)?in)(\d{6})(?:_\d+)?", url)
    if not m:
        return ""

    raw = m.group(1)
    yy, mm, dd = int(raw[:2]), int(raw[2:4]), int(raw[4:6])

    year = 2000 + yy if yy < 50 else 1900 + yy

    try:
        return datetime(year, mm, dd).strftime("%Y-%m-%d")
    except ValueError:
        return ""

### 4. Media outlet extraction

Extracts the media outlet name from the interview title. ECB titles
predominantly follow the pattern `Interview with {Outlet}`, but a small
number of older or non-standard entries use slight variations:

- `Interview with Reuters` — standard case
- `Interview withMadame Figaro` — missing whitespace (HTML artefact)
- `TV interview with ZDF Morgenmagazin` — prefix variant
- `Interview for Reuters` — rare preposition variant

The regex below handles all four cases in a single pass. The optional
`TV` prefix and the `with|for` alternation cover the variants without
over-matching. The `re.IGNORECASE` flag makes the match robust to
capitalisation drift across the 20-year corpus.

Note: `Interview of {Person} with {Outlet}` titles (used for some older
Draghi interviews) are deliberately **not** matched here, because the
grammatical structure places the speaker, not the outlet, after `of`.
These titles fall through to an empty string and are left for manual
review if outlet-level analysis requires them.

Returns `""` when no match is found — consistent with the rest of the
pipeline and safe for CSV round-trips.

In [13]:
def parse_media_outlet(title):
    """
    Extract the media outlet from titles like 'Interview with Reuters'.
    """
    m = re.match(r"(?:TV\s+)?Interview\s*(?:with|for)\s*(.+)", title, re.IGNORECASE)
    return m.group(1).strip() if m else ""

### 5. Text cleaning

The paragraph extraction step is intentionally permissive — it collects every `<p>` tag on the page to remain robust across the ECB's changing HTML structure over two decades. The tradeoff is that non-content elements from the site template are also captured: cookie banners (*"We use functional cookies..."*), legal disclaimers, press-office footers (*"European Central Bank — Directorate General Communications"*), and sidebar elements such as related topic tags.

The cleaning step applies a list of regex patterns to remove these recurring elements. The patterns are phrase-based rather than tied to specific HTML classes or tags, which makes them robust to layout changes across the corpus. The list was compiled empirically from inspected output and covers the boilerplate observed in the current scrape; it is not exhaustive, and periodic checks are advisable if the ECB updates its templates.

This step addresses only the artefacts of the delivery medium. Linguistic preprocessing — stopword removal, lemmatisation, tokenisation, n-gram construction — is left to downstream analyses, where it can be tuned to the requirements of specific methods such as topic modelling, sentiment scoring, or Wordscore classification.

In [15]:
NOISE_PATTERNS = [
    r"Disclaimer.*?(?=\n|$)",
    r"Please note that related topic tags.*?(?=\n|$)",
    r"Reproduction is permitted.*?(?=\n|$)",
    r"Media contacts.*?(?=\n|$)",
    r"Thank you for letting us know.*?(?=\n|$)",
    r"We use functional cookies.*$",
    r"Read our privacy statement.*$",
    r"Learn more about how we use cookies.*$",
    r"Related topics.*?(?=\n|$)",
    r"European Central Bank\s*Directorate General Communications.*$",
]

In [16]:
def clean_text(raw_text):
    """Remove navigation noise, disclaimers, cookie banners."""
    text = raw_text
    for pat in NOISE_PATTERNS:
        text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

### 6. Q&A structure extraction

ECB interview pages present content as a flat sequence of paragraphs, without explicit markup distinguishing interviewer questions from speaker answers. Their separation must therefore be inferred from the text.

The function applies a simple paragraph-level heuristic: a paragraph is treated as a question if it ends with a question mark and is fewer than 600 characters long; otherwise, it is treated as part of the preceding answer. Text appearing before the first detected question is retained as a `[PREAMBLE]` entry rather than discarded.

The 600-character cap is supported by the length distribution of accepted questions in the corpus (99.5th percentile: 513 chars; maximum: 593 chars), placing the threshold above the tail of observed interviewer prompts while remaining below the bulk of long ?-ending spans found inside answer text."

In [18]:
def extract_qa_pairs(paragraphs_text):
    """
    Separate interview text into (question, answer) pairs.
    
    Heuristic: questions in ECB interviews typically end with '?'
    and are relatively short compared to the answers. We split
    on paragraph boundaries and classify each as Q or A.
    """
    paragraphs = [p.strip() for p in paragraphs_text.split("\n") if p.strip()]
    
    qa_pairs = []
    current_q = ""
    current_a_parts = []
    
    for para in paragraphs:        
        is_question = (
            para.strip().endswith("?") and
            len(para) < 600
        )
        
        if is_question:            
            if current_q and current_a_parts:
                qa_pairs.append({
                    "question": current_q,
                    "answer": " ".join(current_a_parts)
                })
            elif current_a_parts and not current_q:
                # Intro text before first question
                qa_pairs.append({
                    "question": "[PREAMBLE]",
                    "answer": " ".join(current_a_parts)
                })
            current_q = para
            current_a_parts = []
        else:
            current_a_parts.append(para)    
   
    if current_q and current_a_parts:
        qa_pairs.append({
            "question": current_q,
            "answer": " ".join(current_a_parts)
        })
    
    return qa_pairs

### 7. Speaker extraction and name normalisation

Speaker identification across a 20-year corpus faces two distinct problems. First, the same person appears under multiple written forms — accents dropped or added (*Vítor Constâncio* / *Vitor Constancio*), informal variants (*Claude Trichet* / *Jean-Claude Trichet*), dropped middle initials (*Philip Lane* / *Philip R. Lane*), and HTML extraction glitches (*Christine LagardeThe*, where the opening of the next paragraph concatenates to the name). Second, the HTML elements carrying the speaker's name have changed repeatedly across the ECB's site history, and the surrounding text ("Video interview with...", "Transcript of...", "BILD interview with...") varies from page to page.

Both problems are addressed by anchoring on the **speaker**, not on the surrounding structure. `CANONICAL_SPEAKERS` is a curated dictionary mapping most observed spelling of every ECB Executive Board member — past and present — to a single canonical form. Because the universe of possible speakers is small and known (roughly 25 lifetime members), matching against this list is more precise than regex-parsing open-ended wrapper phrases or applying fuzzy string matching, which risks collapsing distinct names.

The `extract_speaker` function then searches candidate text blocks in descending order of reliability: the structured `author-details` div, the subtitle classes (`ecb-pressContentSubtitle` and its plural variant, which the original scraper missed), the publication-date element, `<h1>` and `<h2>` headers, and the first few paragraphs of the article. Within each candidate, a regex built from the canonical dictionary — sorted longest-first, so *Philip R. Lane* is tried before *Philip Lane* — locates any known name. A second pass applies a generic *"Mr/Mrs [Name]"* pattern for pre-2007 pages that predate the structured layout, then re-snaps the result to canonical form where possible.

This design makes downstream speaker-stratified analysis (e.g. topic trajectories per president, speaker-level sentiment panels) meaningful: one person corresponds to exactly one label. The dictionary is the single point of maintenance — adding a newly appointed Executive Board member updates both passes simultaneously.

In [20]:
CANONICAL_SPEAKERS = {
    "jean-claude trichet":           "Jean-Claude Trichet",
    "claude trichet":                "Jean-Claude Trichet",
    "mario draghi":                  "Mario Draghi",
    "christine lagarde":             "Christine Lagarde",
    "vitor constancio":              "Vítor Constâncio",
    "vítor constâncio":              "Vítor Constâncio",
    "luis de guindos":               "Luis de Guindos",
    "peter praet":                   "Peter Praet",
    "benoit coeure":                 "Benoît Cœuré",
    "benoît cœuré":                  "Benoît Cœuré",
    "yves mersch":                   "Yves Mersch",
    "sabine lautenschlager":         "Sabine Lautenschläger",
    "sabine lautenschläger":         "Sabine Lautenschläger",
    "philip r. lane":                "Philip R. Lane",
    "philip lane":                   "Philip R. Lane",
    "isabel schnabel":               "Isabel Schnabel",
    "fabio panetta":                 "Fabio Panetta",
    "frank elderson":                "Frank Elderson",
    "piero cipollone":               "Piero Cipollone",
    "lucas papademos":               "Lucas Papademos",
    "jörg asmussen":                 "Jörg Asmussen",
    "jorg asmussen":                 "Jörg Asmussen",
    "lorenzo bini smaghi":           "Lorenzo Bini Smaghi",
    "otmar issing":                  "Otmar Issing",
    "tommaso padoa-schioppa":        "Tommaso Padoa-Schioppa",
    "josé manuel gonzález-páramo":   "José Manuel González-Páramo",
    "jose manuel gonzalez-paramo":   "José Manuel González-Páramo",
    "gertrude tumpel-gugerell":      "Gertrude Tumpel-Gugerell",
}

# Build a regex alternation of canonical names, longest-first so that
# "Philip R. Lane" is tried before "Philip Lane" etc.
_NAME_KEYS = sorted(CANONICAL_SPEAKERS.keys(), key=len, reverse=True)
_NAME_ALTERNATION = "|".join(re.escape(k) for k in _NAME_KEYS)
_NAME_FINDER = re.compile(_NAME_ALTERNATION, flags=re.IGNORECASE)


In [21]:
def extract_speaker(soup):
    """
    Extract the ECB Executive Board member's name from an interview page.

    Strategy (highest-confidence first):
      1. Check `author-details` div — clean structured data when present.
      2. Scan the subtitle / h2 / h1 text for any known canonical name.
         This sidesteps brittle regex-based name parsing entirely:
         we don't need to recognize "Article by", "Video interview with",
         "Transcript of", "BILD interview with", or any other wrapper —
         we just look for the name itself and let the wrapper fall away.
      3. Fall back to a generic "Mr/Mrs [Name]" pattern for very old pages.

    Returns the canonical name as a string, or "" if nothing matches.
    """
    
    candidates = []

    author_el = soup.find(attrs={"class": "author-details"})
    if author_el:
        candidates.append(author_el.get_text(strip=True))


    for cls in ("ecb-pressContentSubtitle", "ecb-pressContentSubtitles"):
        el = soup.find(attrs={"class": cls})
        if el:
            candidates.append(el.get_text(strip=True))


    pub_el = soup.find(attrs={"class": "ecb-pressContentPubDate"})
    if pub_el:
        candidates.append(pub_el.get_text(strip=True))


    for h2 in soup.find_all("h2")[:3]:
        candidates.append(h2.get_text(strip=True))
    h1 = soup.find("h1")
    if h1:
        candidates.append(h1.get_text(strip=True))


    for p in soup.find_all("p")[:3]:
        candidates.append(p.get_text(strip=True))

    for text in candidates:
        if not text:
            continue
        match = _NAME_FINDER.search(text)
        if match:
            key = match.group(0).lower()
            return CANONICAL_SPEAKERS[key]


    for text in candidates:
        if not text:
            continue
        m = re.search(
            r"(?:Mr|Mrs|Ms)\.?\s+"
            r"([\w\-]+(?:\s+[\w\-]+){1,4})"
            r"\s*,",
            text,
        )
        if m:
            raw = m.group(1).strip()

            key = raw.lower()
            if key in CANONICAL_SPEAKERS:
                return CANONICAL_SPEAKERS[key]

            return raw

    return ""


## 8. Interview extraction pipeline

This function implements the core extraction step of the pipeline, transforming a raw ECB interview URL into a structured observation suitable for analysis.

In [23]:
def extract_interview(url):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Title
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""
    
    # Media outlet
    media_outlet = parse_media_outlet(title)
    
    # Date
    date = parse_date_from_url(url)
    
    # Speaker
    speaker = extract_speaker(soup)
    
    # Main content — all <p> tags, then clean
    paragraphs = soup.find_all("p")
    raw_text = "\n".join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True))
    text = clean_text(raw_text)
    
    # Q&A pairs
    qa_pairs = extract_qa_pairs(text)
    
    # Answers only
    answers_only = " ".join(
        qa["answer"] for qa in qa_pairs
        if qa["question"] != "[PREAMBLE]"
    )
    
    return {
        "url": url,
        "date": date,
        "speaker": speaker,
        "media_outlet": media_outlet,
        "title": title,
        "full_text": text,
        "answers_only": answers_only,
        "n_qa_pairs": len(qa_pairs),
        "qa_pairs_json": qa_pairs,
        "word_count": len(text.split()),
    }

### 9. Batch extraction (Running the Scraper)

The extraction function is applied sequentially to every discovered URL. Each successful call returns a structured record (metadata, cleaned text, Q&A pairs, derived counts), which is appended to a single list that forms the final dataset.

Errors are caught at the URL level: if any individual page fails — due to network issues, unexpected HTML, or a parsing exception — the URL and error message are recorded in a separate `errors` list and the loop continues. This isolates per-page failures without interrupting the run and preserves an audit trail for manual inspection afterwards.

A one-second delay between requests (`DELAY = 1`) keeps the load on the ECB's servers low and aligns with standard polite-scraping conventions. Progress is printed per URL, reporting word count and detected speaker as lightweight in-flight sanity checks; a final line summarises how many interviews were collected and how many failed.

On the corpus used here, the loop processed all 590 URLs with zero errors.

In [25]:
data = []
errors = []

total = len(interview_urls)
print(f"Scraping {total} interviews...\n")

for i, url in enumerate(interview_urls, 1):
    print(f"[{i}/{total}] {url[:90]}...", end=" ")
    try:
        record = extract_interview(url)
        data.append(record)
        print(f"OK ({record['word_count']} words, speaker: {record['speaker'] or '?'})")
    except Exception as e:
        print(f"ERROR: {e}")
        errors.append({"url": url, "error": str(e)})
    time.sleep(DELAY)

print(f"\nDone. Collected: {len(data)} | Errors: {len(errors)}")

Scraping 590 interviews...

[1/590] https://www.ecb.europa.eu/press/inter/date/2026/html/ecb.in260323~b893c61107.en.html... OK (2352 words, speaker: Luis de Guindos)
[2/590] https://www.ecb.europa.eu/press/inter/date/2026/html/ecb.in260308~1c03ad3ece.en.html... OK (1707 words, speaker: Christine Lagarde)
[3/590] https://www.ecb.europa.eu/press/inter/date/2026/html/ecb.in260303~768fa10188.en.html... OK (2127 words, speaker: Philip R. Lane)
[4/590] https://www.ecb.europa.eu/press/inter/date/2026/html/ecb.in260221~a2c547fb40.en.html... OK (5860 words, speaker: Christine Lagarde)
[5/590] https://www.ecb.europa.eu/press/inter/date/2026/html/ecb.in260219~297f9872eb.en.html... OK (2441 words, speaker: Frank Elderson)
[6/590] https://www.ecb.europa.eu/press/inter/date/2026/html/ecb.in260210~b0b5032167.en.html... OK (2312 words, speaker: Luis de Guindos)
[7/590] https://www.ecb.europa.eu/press/inter/date/2026/html/ecb.in260208~7abe463638.en.html... OK (1806 words, speaker: Piero Cipollone)
[8/5

## 10. Build the DataFrame

In [41]:
df = pd.DataFrame(data)

# Convert date column
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date", ascending=False).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nSpeakers found:")
print(df["speaker"].value_counts().head(15))
print(f"\nMedia outlets:")
print(df["media_outlet"].value_counts().head(15))

Shape: (590, 10)
Date range: 2004-06-18 00:00:00 to 2026-03-23 00:00:00

Speakers found:
speaker
Benoît Cœuré             87
Jean-Claude Trichet      84
Luis de Guindos          66
Philip R. Lane           58
Isabel Schnabel          56
Christine Lagarde        50
Peter Praet              43
Mario Draghi             27
Sabine Lautenschläger    20
Yves Mersch              19
Fabio Panetta            19
Vítor Constâncio         15
Frank Elderson           13
Piero Cipollone          12
Lucas Papademos           5
Name: count, dtype: int64

Media outlets:
media_outlet
                                  58
Le Monde                          20
Financial Times                   15
Il Sole 24 Ore                    14
Handelsblatt                      14
Bloomberg                         14
Börsen-Zeitung                    13
Les Echos                         12
Süddeutsche Zeitung               12
Die Zeit                          11
Reuters                           11
the Financial Times  

## 11. Data export

The final dataset is exported in two complementary formats to support different use cases and ensure reproducibility **CSV** and **Parquet**

In [44]:
import json
import csv

# --- Main dataset (one row per interview) ---
df_export = df.copy()
df_export["qa_pairs_json"] = df_export["qa_pairs_json"].apply(json.dumps)

# Standard CSV with aggressive quoting (safe for any text content)
df_export.to_csv(
    "ecb_interviews.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
    encoding="utf-8",
)

# Parquet — canonical, typed, lossless
df_export.to_parquet("ecb_interviews.parquet", index=False)

print(f"Saved ecb_interviews.csv ({len(df_export)} rows)")
print(f"Saved ecb_interviews.parquet")

Saved ecb_interviews.csv (590 rows)
Saved ecb_interviews.parquet


## 12. Q&A long-format dataset
In addition to the interview-level dataset, a long-format version is constructed at the question–answer level. This representation expands each interview into multiple observations, where each row corresponds to a single Q&A pair.

In [47]:
qa_rows = []
for _, row in df.iterrows():
    for j, qa in enumerate(row["qa_pairs_json"]):
        qa_rows.append({
            "date": row["date"],
            "speaker": row["speaker"],
            "media_outlet": row["media_outlet"],
            "qa_index": j,
            "question": qa["question"],
            "answer": qa["answer"],
            "url": row["url"],
        })

df_qa = pd.DataFrame(qa_rows)
df_qa.to_csv("ecb_interviews_qa.csv", index=False)

print(f"Saved ecb_interviews_qa.csv ({len(df_qa)} Q&A pairs across {df_qa['url'].nunique()} interviews)")

Saved ecb_interviews_qa.csv (8771 Q&A pairs across 550 interviews)


## Save any errors for review

In [54]:
if errors:
    df_errors = pd.DataFrame(errors)
    df_errors.to_csv("ecb_interviews_errors.csv", index=False)
    print(f"Saved ecb_interviews_errors.csv ({len(errors)} errors)")
else:
    print("DONE!... No errors to save.")

DONE!... No errors to save.
